# CO5430 Melanoma Classification — Baseline Pipeline

**Group 15 | CO5430 Computer Vision**

This notebook implements a baseline binary classification pipeline for melanoma detection using the [Melanoma Cancer Dataset](https://www.kaggle.com/datasets/bhaveshmittal/melanoma-cancer-dataset) from Kaggle.

**Pipeline Overview:**
1. Install dependencies & download dataset
2. Locate train/test directories
3. Imports
4. Transforms & Datasets
5. Class balance check
6. Data loaders
7. Model (EfficientNetV2-S)
8. Loss, Optimizer, Scheduler
9. Early stopping
10. Training loop
11. Loss / Accuracy plots
12. Confusion matrix

Run each cell in order, top to bottom.

## Step 1: Install Kaggle CLI and Download the Dataset

> **Note:** Replace `PASTE_YOUR_NEW_TOKEN_HERE` with your actual Kaggle API token (JSON string), or upload `kaggle.json` manually to `~/.kaggle/kaggle.json`.

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle
!echo "PASTE_YOUR_NEW_TOKEN_HERE" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d bhaveshmittal/melanoma-cancer-dataset -p /content/data --unzip

## Step 2: Find the Actual train/test Folders

Handles any folder nesting that may arise after unzipping.

In [ ]:
import os

def find_dir(root, target_name):
    for dirpath, dirnames, _ in os.walk(root):
        if target_name in dirnames:
            return os.path.join(dirpath, target_name)
    return None

path = find_dir('/content/data', 'train')
valPath = find_dir('/content/data', 'test')

print('train path:', path)
print('test  path:', valPath)

if path is None or valPath is None:
    raise FileNotFoundError("Couldn't locate train/test folders — check !ls /content/data")

## Step 3: Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from warnings import filterwarnings
filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 4: Transforms and Datasets

- **Training transforms:** Random rotation (±20°), random horizontal/vertical flips, resize to 112×112, ImageNet normalization.
- **Validation transforms:** Resize + center crop + normalization only (no augmentation).

In [ ]:
imgSize = 112

transformer = transforms.Compose([
    transforms.RandomRotation(degrees=20),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.Resize(size=(imgSize, imgSize), antialias=True),
    transforms.CenterCrop(size=(imgSize, imgSize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valTransformer = transforms.Compose([
    transforms.Resize(size=(imgSize, imgSize), antialias=True),
    transforms.CenterCrop(size=(imgSize, imgSize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

trainData = datasets.ImageFolder(root=path, transform=transformer)
valData   = datasets.ImageFolder(root=valPath, transform=valTransformer)

print('Classes:', trainData.classes)

## Step 5: Class Balance Check

Computes per-class sample counts. This informs the `pos_weight` used in Step 8 to handle class imbalance.

In [ ]:
from collections import Counter
counts = Counter(trainData.targets)
print({trainData.classes[k]: v for k, v in counts.items()})

## Step 6: Data Loaders

In [ ]:
batchSize = 256

trainLoader = DataLoader(trainData, batch_size=batchSize, shuffle=True,  num_workers=2)
valLoader   = DataLoader(valData,   batch_size=batchSize, shuffle=False, num_workers=2)

## Step 7: Model — EfficientNetV2-S (Transfer Learning)

We replace the final classification head with a single neuron for binary output (`BCEWithLogitsLoss`).

In [ ]:
model = models.efficientnet_v2_s(weights='DEFAULT')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(device)

## Step 8: Loss, Optimizer, and Scheduler

- **Loss:** `BCEWithLogitsLoss` with `pos_weight` to compensate for the benign/malignant class imbalance.
- **Optimizer:** Adam (lr=0.001)
- **Scheduler:** `ReduceLROnPlateau` — halves LR when validation loss plateaus for 3 epochs.

In [ ]:
num_benign    = counts[trainData.class_to_idx['benign']]    if 'benign'    in trainData.class_to_idx else list(counts.values())[0]
num_malignant = counts[trainData.class_to_idx['malignant']] if 'malignant' in trainData.class_to_idx else list(counts.values())[1]
pos_weight = torch.tensor([num_benign / num_malignant]).to(device)

criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = optim.Adam(model.parameters(), lr=0.001)
scheduler  = ReduceLROnPlateau(optimizer, threshold=0.01, factor=0.1, patience=3, min_lr=1e-5)

## Step 9: Early Stopping Variables

In [ ]:
patience        = 5
minDelta        = 0.01
currentPatience = 0
bestLoss        = float('inf')

## Step 10: Training Loop

Uses **mixed precision training** (`autocast` + `GradScaler`) for faster GPU computation.

In [ ]:
scaler = GradScaler()
trainLosses, valLosses, valAccs = [], [], []
epochs = 10

for epoch in range(epochs):
    model.train()
    runningLoss = 0.0
    for inputs, labels in trainLoader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.unsqueeze(1).float()

        optimizer.zero_grad()
        with autocast():
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        runningLoss += loss.item()

    trainLoss = runningLoss / len(trainLoader)
    print(f'Epoch {epoch+1}/{epochs} - Training Loss: {trainLoss:.4f}')
    trainLosses.append(trainLoss)

    model.eval()
    with torch.no_grad():
        valLoss, correct, total = 0.0, 0, 0
        for inputs, labels in valLoader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels  = labels.unsqueeze(1).float()
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            valLoss += loss.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total    += labels.size(0)
            correct  += (predicted == labels).sum().item()

        avgLoss  = valLoss / len(valLoader)
        accuracy = correct / total * 100
        print(f'Validation Loss: {avgLoss:.4f} | Validation Accuracy: {accuracy:.2f}%\n')
        valLosses.append(avgLoss)
        valAccs.append(accuracy)

        if avgLoss < bestLoss - minDelta:
            bestLoss        = avgLoss
            currentPatience = 0
        else:
            currentPatience += 1
            if currentPatience >= patience:
                print('Early stopping triggered.')
                break

        scheduler.step(avgLoss)

## Step 11: Plot Loss / Accuracy Curves

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(trainLosses, label='Training Loss')
plt.plot(valLosses,   label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(valAccs, label='Validation Accuracy', color='green')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.legend(); plt.title('Validation Accuracy')

plt.tight_layout()
plt.show()

## Step 12: Confusion Matrix

In [ ]:
model.eval()
allLabels, allPreds = [], []
with torch.no_grad():
    for inputs, labels in valLoader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels      = labels.unsqueeze(1).float()
        outputs     = model(inputs)
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        allLabels.extend(labels.cpu().numpy())
        allPreds.extend(predictions.cpu().numpy())

allLabels = np.array(allLabels)
allPreds  = np.array(allPreds)
matrix    = confusion_matrix(allLabels, allPreds)

plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Greens',
            xticklabels=valData.classes,
            yticklabels=valData.classes,
            cbar=False)
plt.title('Confusion Matrix — Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()